In [1]:
#loading keys

from dotenv import load_dotenv
load_dotenv(override=True)

True

In [2]:
import os 
openai_api_key = os.getenv('OPENAI_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")

OpenAI API Key exists and begins sk-proj-


In [4]:
from openai import OpenAI

client = OpenAI()

In [6]:
message = [
    {
        "role" : "user",
        "content" : "Tell me a fun fact"
    }
]

response = client.chat.completions.create(model = "gpt-5.4-nano",messages =message)

reply = response.choices[0].message.content
print(reply)

Octopuses have three hearts—two pump blood to their gills, and the third pumps blood to the rest of their body. Also, when an octopus swims, the one pumping heart slows down, so they often prefer crawling over swimming.


In [7]:
from openai import OpenAI

client = OpenAI()

response = client.responses.create(
    model="gpt-5.4-nano",
    input="Write a one-sentence bedtime story about a unicorn.",
)

print(response.output_text)

In a moonlit meadow, a gentle unicorn tiptoed through the dew, sprinkling stardust on every sleeping flower until the whole night yawned softly into dreams.


## chain two calls

In [10]:
question_prompt = "Please propose a hard, challenging question to assess someone's IQ. Respond only with the question."

question_response = client.responses.create(

    model = "gpt-5.4-mini",
    input = question_prompt
)

question = question_response.output_text
print(question)

A clock shows 3:15. What is the exact angle between the hour hand and the minute hand, and why does your method still work if the clock’s hour hand moves continuously rather than in discrete steps?


In [11]:
answer_response = client.responses.create(
    model="gpt-5.4-mini",
    input=question,
)

answer = answer_response.output_text
print(answer)

At **3:15**, the angle between the hour and minute hands is:

### 1) Minute hand
At 15 minutes, the minute hand points at the **3**, which is:
- \(15 \times 6^\circ = 90^\circ\) from 12.

### 2) Hour hand
The hour hand is at 3, but it has also moved a bit past 3 because 15 minutes have passed.
- Each hour mark is \(30^\circ\)
- In 15 minutes, the hour hand moves \(\frac{15}{60} \times 30^\circ = 7.5^\circ\)

So the hour hand is at:
- \(3 \times 30^\circ + 7.5^\circ = 97.5^\circ\)

### 3) Angle between them
\[
97.5^\circ - 90^\circ = 7.5^\circ
\]

## Answer:
**The exact angle is \(7.5^\circ\).**

## Why this still works with a continuously moving hour hand
This method works because it already accounts for continuous motion.

- The minute hand moves continuously, so at 15 minutes it is exactly at \(90^\circ\).
- The hour hand does **not** jump from hour to hour; it moves smoothly at a constant rate.
- So when we calculate the hour hand’s position as
  \[
  30^\circ \times \text{hour} + 0

In [12]:
from IPython.display import Markdown,display

display(Markdown(answer))

At **3:15**, the angle between the hour and minute hands is:

### 1) Minute hand
At 15 minutes, the minute hand points at the **3**, which is:
- \(15 \times 6^\circ = 90^\circ\) from 12.

### 2) Hour hand
The hour hand is at 3, but it has also moved a bit past 3 because 15 minutes have passed.
- Each hour mark is \(30^\circ\)
- In 15 minutes, the hour hand moves \(\frac{15}{60} \times 30^\circ = 7.5^\circ\)

So the hour hand is at:
- \(3 \times 30^\circ + 7.5^\circ = 97.5^\circ\)

### 3) Angle between them
\[
97.5^\circ - 90^\circ = 7.5^\circ
\]

## Answer:
**The exact angle is \(7.5^\circ\).**

## Why this still works with a continuously moving hour hand
This method works because it already accounts for continuous motion.

- The minute hand moves continuously, so at 15 minutes it is exactly at \(90^\circ\).
- The hour hand does **not** jump from hour to hour; it moves smoothly at a constant rate.
- So when we calculate the hour hand’s position as
  \[
  30^\circ \times \text{hour} + 0.5^\circ \times \text{minutes},
  \]
  we are measuring its exact continuous position at that moment.

So the formula gives the true angle whether the hour hand is treated as moving smoothly or just approximated in steps.

In [13]:
judge_prompt = f"""
Here is a question:
{question}

And here is a possible answer that might be correct or incorrect:
{answer}

Please evaluate if the answer is correct or incorrect.
"""

print(judge_prompt)


Here is a question:
A clock shows 3:15. What is the exact angle between the hour hand and the minute hand, and why does your method still work if the clock’s hour hand moves continuously rather than in discrete steps?

And here is a possible answer that might be correct or incorrect:
At **3:15**, the angle between the hour and minute hands is:

### 1) Minute hand
At 15 minutes, the minute hand points at the **3**, which is:
- \(15 \times 6^\circ = 90^\circ\) from 12.

### 2) Hour hand
The hour hand is at 3, but it has also moved a bit past 3 because 15 minutes have passed.
- Each hour mark is \(30^\circ\)
- In 15 minutes, the hour hand moves \(\frac{15}{60} \times 30^\circ = 7.5^\circ\)

So the hour hand is at:
- \(3 \times 30^\circ + 7.5^\circ = 97.5^\circ\)

### 3) Angle between them
\[
97.5^\circ - 90^\circ = 7.5^\circ
\]

## Answer:
**The exact angle is \(7.5^\circ\).**

## Why this still works with a continuously moving hour hand
This method works because it already accounts for 

In [14]:
judge_response = client.responses.create(
    model="gpt-5.4",
    input=judge_prompt,
)

print(judge_response.output_text)

The answer is **correct**.

Why:
- Minute hand at 15 minutes: \(15 \times 6^\circ = 90^\circ\)
- Hour hand at 3:15:  
  - At 3:00 it is at \(3 \times 30^\circ = 90^\circ\)
  - It moves \(30^\circ\) per hour, so in 15 minutes it moves \(\frac{15}{60}\times 30^\circ = 7.5^\circ\)
  - So its position is \(90^\circ + 7.5^\circ = 97.5^\circ\)

Thus the angle between the hands is:
\[
|97.5^\circ - 90^\circ| = 7.5^\circ
\]

The explanation about continuous motion is also correct:
- The hour hand moves continuously, not in jumps.
- The formula \(30h + 0.5m\) degrees exactly models that continuous motion.
- So the method remains valid for a real clock.

**Verdict: Correct.**


In [ ]:
git add -A
git commit -m "feat: day 1 llm-as-judge evaluation"
git push